In [ ]:
!pip install -q openai==1.12.0
!pip install -q pandas numpy matplotlib seaborn
!pip install -q python-dotenv
!pip install -q sqlite3  # Built-in, but just to confirm
!pip install -q requests beautifulsoup4
!pip install -q smtplib  # Built-in
!pip install -q streamlit pyngrok  # For dashboard
!pip install -q plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 4.7 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3
ERROR: Could not find a version that satisfies the requirement smtplib (from versions: none)
ERROR: No matching distribution found for smtplib
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.2 MB/s eta 0:00:00


In [ ]:
!pip install -q openai==1.12.0
!pip install -q pandas numpy matplotlib seaborn
!pip install -q python-dotenv
!pip install -q requests beautifulsoup4
!pip install -q streamlit pyngrok
!pip install -q plotly
!pip install -q razorpay  # For payments
!pip install -q linkedin-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.9/176.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 47.0 MB/s eta 0:00:00


In [ ]:
# ==============================================
# CELL 3: DATABASE SETUP (USING BUILT-IN SQLITE3)
# ==============================================

class Database:
    """SQLite database for storing all data"""

    def __init__(self, db_name="eventflow.db"):
        self.db_name = db_name
        self.conn = None
        self.cursor = None
        self.connect()
        self.create_tables()

    def connect(self):
        """Connect to database"""
        self.conn = sqlite3.connect(self.db_name)
        self.cursor = self.conn.cursor()
        print(f"✅ Connected to {self.db_name}")

    def create_tables(self):
        """Create all necessary tables"""

        # Prospects table
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS prospects (
                id TEXT PRIMARY KEY,
                name TEXT NOT NULL,
                title TEXT,
                company TEXT,
                industry TEXT,
                location TEXT,
                email TEXT,
                phone TEXT,
                company_size TEXT,
                revenue TEXT,
                engagement_score INTEGER,
                source TEXT,
                created_at TIMESTAMP,
                updated_at TIMESTAMP
            )
        """)

        # Campaigns table
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS campaigns (
                id TEXT PRIMARY KEY,
                name TEXT NOT NULL,
                event_type TEXT,
                start_date TIMESTAMP,
                end_date TIMESTAMP,
                status TEXT,
                budget REAL,
                target_audience TEXT,
                results TEXT,
                created_at TIMESTAMP
            )
        """)

        # Messages table
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id TEXT PRIMARY KEY,
                prospect_id TEXT,
                campaign_id TEXT,
                message TEXT,
                channel TEXT,
                sent_at TIMESTAMP,
                opened_at TIMESTAMP,
                replied_at TIMESTAMP,
                status TEXT,
                response TEXT,
                FOREIGN KEY (prospect_id) REFERENCES prospects (id),
                FOREIGN KEY (campaign_id) REFERENCES campaigns (id)
            )
        """)

        # Conversations table
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS conversations (
                id TEXT PRIMARY KEY,
                prospect_id TEXT,
                message_id TEXT,
                sender TEXT,
                content TEXT,
                sentiment REAL,
                intent TEXT,
                timestamp TIMESTAMP,
                FOREIGN KEY (prospect_id) REFERENCES prospects (id)
            )
        """)

        # Payments table
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS payments (
                id TEXT PRIMARY KEY,
                prospect_id TEXT,
                campaign_id TEXT,
                amount REAL,
                currency TEXT,
                status TEXT,
                payment_method TEXT,
                transaction_id TEXT,
                payment_link TEXT,
                created_at TIMESTAMP,
                completed_at TIMESTAMP,
                FOREIGN KEY (prospect_id) REFERENCES prospects (id)
            )
        """)

        # Analytics table
        self.cursor.execute("""
            CREATE TABLE IF NOT EXISTS analytics (
                id TEXT PRIMARY KEY,
                campaign_id TEXT,
                metric_name TEXT,
                metric_value REAL,
                dimension TEXT,
                timestamp TIMESTAMP
            )
        """)

        self.conn.commit()
        print("✅ All tables created/verified")

    def save_prospect(self, prospect):
        """Save prospect to database"""
        try:
            prospect_id = prospect.get('id', f"P{int(time.time())}{random.randint(100,999)}")
            self.cursor.execute("""
                INSERT OR REPLACE INTO prospects
                (id, name, title, company, industry, location, email,
                 company_size, revenue, engagement_score, source, created_at, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                prospect_id,
                prospect.get('name', ''),
                prospect.get('title', ''),
                prospect.get('company', ''),
                prospect.get('industry', ''),
                prospect.get('location', ''),
                prospect.get('email', ''),
                prospect.get('company_size', ''),
                prospect.get('revenue', ''),
                prospect.get('engagement_score', 50),
                prospect.get('source', 'manual'),
                datetime.now(),
                datetime.now()
            ))
            self.conn.commit()
            return prospect_id
        except Exception as e:
            print(f"❌ Error saving prospect: {e}")
            return None

    def save_campaign(self, campaign):
        """Save campaign to database"""
        try:
            campaign_id = campaign.get('id', f"CAMP{int(time.time())}")
            self.cursor.execute("""
                INSERT OR REPLACE INTO campaigns
                (id, name, event_type, start_date, status, budget, target_audience, created_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                campaign_id,
                campaign.get('name', ''),
                campaign.get('event_type', ''),
                campaign.get('start_date', datetime.now()),
                campaign.get('status', 'active'),
                campaign.get('budget', 50000),
                json.dumps(campaign.get('target_audience', {})),
                datetime.now()
            ))
            self.conn.commit()
            return campaign_id
        except Exception as e:
            print(f"❌ Error saving campaign: {e}")
            return None

    def save_message(self, prospect_id, campaign_id, message, channel='email'):
        """Save sent message"""
        try:
            msg_id = f"MSG{int(time.time())}{random.randint(100,999)}"
            self.cursor.execute("""
                INSERT INTO messages (id, prospect_id, campaign_id, message, channel, sent_at, status)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (msg_id, prospect_id, campaign_id, message, channel, datetime.now(), 'sent'))
            self.conn.commit()
            return msg_id
        except Exception as e:
            print(f"❌ Error saving message: {e}")
            return None

    def save_payment(self, prospect_id, campaign_id, amount, payment_link=None):
        """Save payment record"""
        try:
            payment_id = f"PAY{int(time.time())}{random.randint(100,999)}"
            self.cursor.execute("""
                INSERT INTO payments (id, prospect_id, campaign_id, amount, currency, status, payment_link, created_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (payment_id, prospect_id, campaign_id, amount, 'INR', 'pending', payment_link, datetime.now()))
            self.conn.commit()
            return payment_id
        except Exception as e:
            print(f"❌ Error saving payment: {e}")
            return None

    def update_payment_status(self, payment_id, status, transaction_id=None):
        """Update payment status"""
        try:
            self.cursor.execute("""
                UPDATE payments
                SET status = ?, transaction_id = ?, completed_at = ?
                WHERE id = ?
            """, (status, transaction_id, datetime.now(), payment_id))
            self.conn.commit()
            return True
        except Exception as e:
            print(f"❌ Error updating payment: {e}")
            return False

    def get_campaign_stats(self, campaign_id):
        """Get campaign statistics"""
        try:
            self.cursor.execute("""
                SELECT
                    COUNT(DISTINCT m.prospect_id) as total_contacted,
                    COUNT(CASE WHEN m.replied_at IS NOT NULL THEN 1 END) as total_replies,
                    COUNT(DISTINCT p.id) as total_prospects,
                    IFNULL(SUM(pay.amount), 0) as total_revenue,
                    COUNT(CASE WHEN pay.status = 'completed' THEN 1 END) as total_payments
                FROM campaigns c
                LEFT JOIN messages m ON c.id = m.campaign_id
                LEFT JOIN prospects p ON m.prospect_id = p.id
                LEFT JOIN payments pay ON p.id = pay.prospect_id AND pay.campaign_id = c.id
                WHERE c.id = ?
            """, (campaign_id,))

            result = self.cursor.fetchone()
            return {
                'total_contacted': result[0] or 0,
                'total_replies': result[1] or 0,
                'total_prospects': result[2] or 0,
                'total_revenue': result[3] or 0,
                'total_payments': result[4] or 0
            }
        except Exception as e:
            print(f"❌ Error getting stats: {e}")
            return {}

    def get_all_prospects(self, limit=100):
        """Get all prospects"""
        self.cursor.execute("SELECT * FROM prospects ORDER BY engagement_score DESC LIMIT ?", (limit,))
        return self.cursor.fetchall()

    def close(self):
        """Close database connection"""
        if self.conn:
            self.conn.close()
            print("✅ Database connection closed")

# Create database instance
db = Database()
print("✅ Database system ready!")

NameError: name 'sqlite3' is not defined

In [ ]:
# ==============================================
# CELL 2: IMPORTS (FIXED - WITH ALL IMPORTS)
# ==============================================

# External packages
import openai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
import razorpay

# IMPORTANT: Built-in modules MUST be imported
import sqlite3  # ✅ YEH IMPORT KARNA HI HOGA
import smtplib
import email
import json
import os
import random
import time
import hashlib
import hmac
import getpass
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import threading
import queue
from collections import defaultdict

# LinkedIn (optional)
try:
    from linkedin_api import Linkedin
    LINKEDIN_AVAILABLE = True
    print("✅ LinkedIn API available")
except:
    LINKEDIN_AVAILABLE = False
    print("ℹ️ LinkedIn API optional - continuing without it")

print("\n" + "="*60)
print("📦 IMPORT STATUS:")
print(f"✅ sqlite3: {sqlite3.version}")
print(f"✅ smtplib: Available")
print(f"✅ Pandas: {pd.__version__}")
print(f"✅ NumPy: {np.__version__}")
print(f"✅ OpenAI: {'Available' if hasattr(openai, '__version__') else 'Installed'}")
print("="*60 + "\n")

print("🎯 All imports completed successfully!")

✅ LinkedIn API available

📦 IMPORT STATUS:
✅ sqlite3: 2.6.0
✅ smtplib: Available
✅ Pandas: 2.2.2
✅ NumPy: 2.0.2
✅ OpenAI: Available

🎯 All imports completed successfully!


/tmp/ipykernel_363/2360058056.py:47: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(f"✅ sqlite3: {sqlite3.version}")


In [ ]:
# ==============================================
# CELL 3: DATABASE SETUP (WITH PROPER IMPORTS)
# ==============================================

class Database:
    """SQLite database for storing all data"""

    def __init__(self, db_name="eventflow.db"):
        self.db_name = db_name
        self.conn = None
        self.cursor = None
        self.connect()
        self.create_tables()

    def connect(self):
        """Connect to database"""
        try:
            self.conn = sqlite3.connect(self.db_name)
            self.cursor = self.conn.cursor()
            print(f"✅ Connected to {self.db_name}")
        except Exception as e:
            print(f"❌ Connection failed: {e}")
            raise

    def create_tables(self):
        """Create all necessary tables"""
        try:
            # Prospects table
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS prospects (
                    id TEXT PRIMARY KEY,
                    name TEXT NOT NULL,
                    title TEXT,
                    company TEXT,
                    industry TEXT,
                    location TEXT,
                    email TEXT,
                    phone TEXT,
                    company_size TEXT,
                    revenue TEXT,
                    engagement_score INTEGER,
                    source TEXT,
                    created_at TIMESTAMP,
                    updated_at TIMESTAMP
                )
            """)

            # Campaigns table
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS campaigns (
                    id TEXT PRIMARY KEY,
                    name TEXT NOT NULL,
                    event_type TEXT,
                    start_date TIMESTAMP,
                    end_date TIMESTAMP,
                    status TEXT,
                    budget REAL,
                    target_audience TEXT,
                    results TEXT,
                    created_at TIMESTAMP
                )
            """)

            # Messages table
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS messages (
                    id TEXT PRIMARY KEY,
                    prospect_id TEXT,
                    campaign_id TEXT,
                    message TEXT,
                    channel TEXT,
                    sent_at TIMESTAMP,
                    opened_at TIMESTAMP,
                    replied_at TIMESTAMP,
                    status TEXT,
                    response TEXT,
                    FOREIGN KEY (prospect_id) REFERENCES prospects (id),
                    FOREIGN KEY (campaign_id) REFERENCES campaigns (id)
                )
            """)

            # Conversations table
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id TEXT PRIMARY KEY,
                    prospect_id TEXT,
                    message_id TEXT,
                    sender TEXT,
                    content TEXT,
                    sentiment REAL,
                    intent TEXT,
                    timestamp TIMESTAMP,
                    FOREIGN KEY (prospect_id) REFERENCES prospects (id)
                )
            """)

            # Payments table
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS payments (
                    id TEXT PRIMARY KEY,
                    prospect_id TEXT,
                    campaign_id TEXT,
                    amount REAL,
                    currency TEXT,
                    status TEXT,
                    payment_method TEXT,
                    transaction_id TEXT,
                    payment_link TEXT,
                    created_at TIMESTAMP,
                    completed_at TIMESTAMP,
                    FOREIGN KEY (prospect_id) REFERENCES prospects (id)
                )
            """)

            # Analytics table
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS analytics (
                    id TEXT PRIMARY KEY,
                    campaign_id TEXT,
                    metric_name TEXT,
                    metric_value REAL,
                    dimension TEXT,
                    timestamp TIMESTAMP
                )
            """)

            self.conn.commit()
            print("✅ All tables created/verified")

        except Exception as e:
            print(f"❌ Table creation failed: {e}")
            raise

    def save_prospect(self, prospect):
        """Save prospect to database"""
        try:
            prospect_id = prospect.get('id', f"P{int(time.time())}{random.randint(100,999)}")
            self.cursor.execute("""
                INSERT OR REPLACE INTO prospects
                (id, name, title, company, industry, location, email,
                 company_size, revenue, engagement_score, source, created_at, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                prospect_id,
                prospect.get('name', ''),
                prospect.get('title', ''),
                prospect.get('company', ''),
                prospect.get('industry', ''),
                prospect.get('location', ''),
                prospect.get('email', ''),
                prospect.get('company_size', ''),
                prospect.get('revenue', ''),
                prospect.get('engagement_score', 50),
                prospect.get('source', 'manual'),
                datetime.now(),
                datetime.now()
            ))
            self.conn.commit()
            return prospect_id
        except Exception as e:
            print(f"❌ Error saving prospect: {e}")
            return None

    def save_campaign(self, campaign):
        """Save campaign to database"""
        try:
            campaign_id = campaign.get('id', f"CAMP{int(time.time())}")
            self.cursor.execute("""
                INSERT OR REPLACE INTO campaigns
                (id, name, event_type, start_date, status, budget, target_audience, created_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                campaign_id,
                campaign.get('name', ''),
                campaign.get('event_type', ''),
                campaign.get('start_date', datetime.now()),
                campaign.get('status', 'active'),
                campaign.get('budget', 50000),
                json.dumps(campaign.get('target_audience', {})),
                datetime.now()
            ))
            self.conn.commit()
            return campaign_id
        except Exception as e:
            print(f"❌ Error saving campaign: {e}")
            return None

    def save_message(self, prospect_id, campaign_id, message, channel='email'):
        """Save sent message"""
        try:
            msg_id = f"MSG{int(time.time())}{random.randint(100,999)}"
            self.cursor.execute("""
                INSERT INTO messages (id, prospect_id, campaign_id, message, channel, sent_at, status)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (msg_id, prospect_id, campaign_id, message, channel, datetime.now(), 'sent'))
            self.conn.commit()
            return msg_id
        except Exception as e:
            print(f"❌ Error saving message: {e}")
            return None

    def save_payment(self, prospect_id, campaign_id, amount, payment_link=None):
        """Save payment record"""
        try:
            payment_id = f"PAY{int(time.time())}{random.randint(100,999)}"
            self.cursor.execute("""
                INSERT INTO payments (id, prospect_id, campaign_id, amount, currency, status, payment_link, created_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (payment_id, prospect_id, campaign_id, amount, 'INR', 'pending', payment_link, datetime.now()))
            self.conn.commit()
            return payment_id
        except Exception as e:
            print(f"❌ Error saving payment: {e}")
            return None

    def update_payment_status(self, payment_id, status, transaction_id=None):
        """Update payment status"""
        try:
            self.cursor.execute("""
                UPDATE payments
                SET status = ?, transaction_id = ?, completed_at = ?
                WHERE id = ?
            """, (status, transaction_id, datetime.now(), payment_id))
            self.conn.commit()
            return True
        except Exception as e:
            print(f"❌ Error updating payment: {e}")
            return False

    def get_campaign_stats(self, campaign_id):
        """Get campaign statistics"""
        try:
            self.cursor.execute("""
                SELECT
                    COUNT(DISTINCT m.prospect_id) as total_contacted,
                    COUNT(CASE WHEN m.replied_at IS NOT NULL THEN 1 END) as total_replies,
                    COUNT(DISTINCT p.id) as total_prospects,
                    IFNULL(SUM(pay.amount), 0) as total_revenue,
                    COUNT(CASE WHEN pay.status = 'completed' THEN 1 END) as total_payments
                FROM campaigns c
                LEFT JOIN messages m ON c.id = m.campaign_id
                LEFT JOIN prospects p ON m.prospect_id = p.id
                LEFT JOIN payments pay ON p.id = pay.prospect_id AND pay.campaign_id = c.id
                WHERE c.id = ?
            """, (campaign_id,))

            result = self.cursor.fetchone()
            return {
                'total_contacted': result[0] or 0,
                'total_replies': result[1] or 0,
                'total_prospects': result[2] or 0,
                'total_revenue': result[3] or 0,
                'total_payments': result[4] or 0
            }
        except Exception as e:
            print(f"❌ Error getting stats: {e}")
            return {}

    def get_all_prospects(self, limit=100):
        """Get all prospects"""
        self.cursor.execute("SELECT * FROM prospects ORDER BY engagement_score DESC LIMIT ?", (limit,))
        return self.cursor.fetchall()

    def get_prospects_by_criteria(self, criteria):
        """Get prospects matching criteria"""
        query = "SELECT * FROM prospects WHERE 1=1"
        params = []

        if 'industry' in criteria:
            placeholders = ','.join(['?' for _ in criteria['industry']])
            query += f" AND industry IN ({placeholders})"
            params.extend(criteria['industry'])

        if 'location' in criteria:
            placeholders = ','.join(['?' for _ in criteria['location']])
            query += f" AND location IN ({placeholders})"
            params.extend(criteria['location'])

        if 'min_score' in criteria:
            query += " AND engagement_score >= ?"
            params.append(criteria['min_score'])

        query += " ORDER BY engagement_score DESC"

        self.cursor.execute(query, params)
        return self.cursor.fetchall()

    def close(self):
        """Close database connection"""
        if self.conn:
            self.conn.close()
            print("✅ Database connection closed")

# Create database instance
print("\n" + "="*60)
print("🗄️ INITIALIZING DATABASE")
print("="*60)

db = Database()
print("✅ Database system ready!")


🗄️ INITIALIZING DATABASE
✅ Connected to eventflow.db
✅ All tables created/verified
✅ Database system ready!


In [ ]:
# ==============================================
# CELL 4: TEST DATABASE
# ==============================================

print("\n" + "="*60)
print("🧪 TESTING DATABASE OPERATIONS")
print("="*60)

# Test prospect
test_prospect = {
    'name': 'Rahul Sharma',
    'title': 'CEO',
    'company': 'TechStart India',
    'industry': 'SaaS',
    'location': 'Mumbai',
    'email': 'rahul@techstart.in',
    'engagement_score': 85,
    'source': 'test'
}

# Save to database
prospect_id = db.save_prospect(test_prospect)
print(f"\n✅ Test prospect saved with ID: {prospect_id}")

# Get all prospects
all_prospects = db.get_all_prospects()
print(f"📊 Total prospects in DB: {len(all_prospects)}")

# Show first prospect
if all_prospects:
    print("\n📋 First prospect in database:")
    print(f"   Name: {all_prospects[0][1]}")
    print(f"   Title: {all_prospects[0][2]}")
    print(f"   Company: {all_prospects[0][3]}")
    print(f"   Industry: {all_prospects[0][4]}")
    print(f"   Location: {all_prospects[0][5]}")
    print(f"   Email: {all_prospects[0][6]}")
    print(f"   Engagement Score: {all_prospects[0][10]}")

print("\n✅ Database test complete!")


🧪 TESTING DATABASE OPERATIONS

✅ Test prospect saved with ID: P1772639687253
📊 Total prospects in DB: 1

📋 First prospect in database:
   Name: Rahul Sharma
   Title: CEO
   Company: TechStart India
   Industry: SaaS
   Location: Mumbai
   Email: rahul@techstart.in
   Engagement Score: 85

✅ Database test complete!


/tmp/ipykernel_363/777518129.py:138: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.cursor.execute("""


In [ ]:
# ==============================================
# CELL 5: INDIAN PROSPECTS DATA
# ==============================================

print("\n" + "="*60)
print("🇮🇳 LOADING INDIAN PROSPECTS")
print("="*60)

INDIAN_PROSPECTS = [
    {
        "name": "Rahul Sharma",
        "title": "CEO",
        "company": "TechStart India",
        "industry": "SaaS",
        "location": "Mumbai",
        "email": "rahul@techstart.in",
        "phone": "+91 98765 43210",
        "company_size": "50-200",
        "revenue": "₹5Cr",
        "pain_points": ["High customer acquisition cost", "Need better lead generation", "Event marketing not working"],
        "engagement_score": 85
    },
    {
        "name": "Priya Patel",
        "title": "Founder",
        "company": "AI Solutions",
        "industry": "AI/ML",
        "location": "Bangalore",
        "email": "priya@aisolutions.in",
        "phone": "+91 98765 43211",
        "company_size": "10-50",
        "revenue": "₹2Cr",
        "pain_points": ["Need more clients", "Building brand awareness", "Finding right partners"],
        "engagement_score": 90
    },
    {
        "name": "Amit Kumar",
        "title": "VP Sales",
        "company": "GrowthCorp",
        "industry": "Enterprise SaaS",
        "location": "Delhi NCR",
        "email": "amit@growthcorp.in",
        "phone": "+91 98765 43212",
        "company_size": "200-500",
        "revenue": "₹20Cr",
        "pain_points": ["Sales team missing targets", "Low conversion rates", "Need sales training"],
        "engagement_score": 75
    },
    {
        "name": "Neha Singh",
        "title": "Director",
        "company": "InnovateLabs",
        "industry": "Technology",
        "location": "Pune",
        "email": "neha@innovatelabs.in",
        "phone": "+91 98765 43213",
        "company_size": "100-250",
        "revenue": "₹15Cr",
        "pain_points": ["Innovation stagnation", "Talent retention", "Market expansion"],
        "engagement_score": 80
    },
    {
        "name": "Vikram Reddy",
        "title": "CTO",
        "company": "CloudNative",
        "industry": "Cloud Computing",
        "location": "Hyderabad",
        "email": "vikram@cloudnative.in",
        "phone": "+91 98765 43214",
        "company_size": "50-150",
        "revenue": "₹8Cr",
        "pain_points": ["Technical debt", "Scaling infrastructure", "Cloud cost optimization"],
        "engagement_score": 70
    },
    {
        "name": "Anjali Mehta",
        "title": "Marketing Head",
        "company": "BrandBoost",
        "industry": "Marketing",
        "location": "Mumbai",
        "email": "anjali@brandboost.in",
        "phone": "+91 98765 43215",
        "company_size": "30-100",
        "revenue": "₹3Cr",
        "pain_points": ["ROI measurement", "Lead quality issues", "Content marketing ineffective"],
        "engagement_score": 65
    },
    {
        "name": "Suresh Iyer",
        "title": "Product Manager",
        "company": "FinTech Innovations",
        "industry": "FinTech",
        "location": "Bangalore",
        "email": "suresh@fintech.in",
        "phone": "+91 98765 43216",
        "company_size": "100-300",
        "revenue": "₹12Cr",
        "pain_points": ["Product-market fit", "User adoption", "Feature prioritization"],
        "engagement_score": 72
    },
    {
        "name": "Deepa Krishnan",
        "title": "HR Director",
        "company": "PeopleFirst",
        "industry": "HR Tech",
        "location": "Chennai",
        "email": "deepa@peoplefirst.in",
        "phone": "+91 98765 43217",
        "company_size": "50-150",
        "revenue": "₹4Cr",
        "pain_points": ["Talent acquisition", "Employee retention", "Company culture"],
        "engagement_score": 68
    },
    {
        "name": "Rajesh Gupta",
        "title": "CEO",
        "company": "WebX Solutions",
        "industry": "Technology",
        "location": "Gurgaon",
        "email": "rajesh@webx.in",
        "phone": "+91 98765 43218",
        "company_size": "20-80",
        "revenue": "₹2.5Cr",
        "pain_points": ["Scaling challenges", "Finding right investors", "Team building"],
        "engagement_score": 78
    },
    {
        "name": "Kavita Krishnan",
        "title": "Founder",
        "company": "DataMatic",
        "industry": "AI/ML",
        "location": "Chennai",
        "email": "kavita@datamatic.in",
        "phone": "+91 98765 43219",
        "company_size": "15-50",
        "revenue": "₹1.5Cr",
        "pain_points": ["Product development", "Market validation", "Early customers"],
        "engagement_score": 82
    }
]

print(f"✅ Loaded {len(INDIAN_PROSPECTS)} Indian prospects")

# Save to database
print("\n💾 Saving prospects to database...")
saved_count = 0
for prospect in INDIAN_PROSPECTS:
    if db.save_prospect(prospect):
        saved_count += 1

print(f"✅ Saved {saved_count} prospects to database")

# Verify
total = len(db.get_all_prospects())
print(f"📊 Total prospects in database: {total}")


🇮🇳 LOADING INDIAN PROSPECTS
✅ Loaded 10 Indian prospects

💾 Saving prospects to database...
✅ Saved 10 prospects to database
📊 Total prospects in database: 11


/tmp/ipykernel_363/777518129.py:138: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.cursor.execute("""


In [ ]:
# ==============================================
# CELL 6: EVENTS CATALOG
# ==============================================

print("\n" + "="*60)
print("📅 LOADING EVENTS CATALOG")
print("="+

SyntaxError: incomplete input (3633857801.py, line 7)

In [ ]:
# ==============================================
# CELL 6: EVENTS CATALOG (COMPLETE)
# ==============================================

print("\n" + "="*60)
print("📅 LOADING EVENTS CATALOG")
print("="*60)

EVENTS = {
    "conference": {
        "name": "AI & SaaS Growth Summit 2024",
        "price": 4999,
        "early_bird": 3999,
        "date": "15-16 June 2024",
        "venue": "Mumbai Convention Center, Bandra Kurla Complex",
        "capacity": 1000,
        "speakers": [
            "Kunal Shah - CRED Founder",
            "Girish Mathrubootham - Freshworks CEO",
            "Nithin Kamath - Zerodha Founder",
            "Falguni Nayar - Nykaa Founder"
        ],
        "topics": [
            "Scaling SaaS in India",
            "AI-powered growth strategies",
            "Fundraising in 2024",
            "Building unicorn culture"
        ],
        "target_industries": ["SaaS", "AI/ML", "Technology", "FinTech"],
        "value_prop": "India's biggest SaaS & AI conference with 50+ successful founders sharing actionable insights"
    },
    "workshop": {
        "name": "Growth Hacking Masterclass",
        "price": 9999,
        "early_bird": 7999,
        "date": "22-23 June 2024",
        "venue": "Bangalore International Tech Park",
        "capacity": 200,
        "speakers": [
            "Anand Rajaraman - Walmart Labs",
            "Kunal Shah - CRED",
            "Tanmay Bhat - Content creator"
        ],
        "topics": [
            "Viral marketing strategies",
            "Conversion rate optimization",
            "Retention hacks",
            "Referral programs"
        ],
        "target_industries": ["SaaS", "Technology", "Marketing", "E-commerce"],
        "value_prop": "Hands-on workshop: Double your revenue in 90 days using proven growth frameworks"
    },
    "mastermind": {
        "name": "CEO Peer Group (12-Month Program)",
        "price": 49999,
        "early_bird": 44999,
        "date": "Starts July 2024 (Monthly for 12 months)",
        "venue": "Multiple cities (Mumbai, Bangalore, Delhi)",
        "capacity": 50,
        "speakers": [
            "Deep Kalra - MakeMyTrip Founder",
            "Bhavish Aggarwal - Ola Founder",
            "Ritesh Agarwal - OYO Founder"
        ],
        "topics": [
            "Quarterly strategy sessions",
            "1:1 mentoring",
            "Peer problem solving",
            "Investor connections"
        ],
        "target_industries": ["SaaS", "Enterprise SaaS", "FinTech"],
        "value_prop": "Exclusive 12-month program for founders scaling from ₹10Cr to ₹100Cr revenue"
    },
    "networking": {
        "name": "Startup Founders Connect",
        "price": 999,
        "early_bird": 699,
        "date": "5 June 2024",
        "venue": "Delhi NCR (Cyber Hub, Gurgaon)",
        "capacity": 300,
        "speakers": [
            "Local founders & VCs",
            "Industry experts"
        ],
        "topics": [
            "Speed networking",
            "Investor meet & greet",
            "Founder stories",
            "Partnership opportunities"
        ],
        "target_industries": ["SaaS", "Technology", "FinTech", "Marketing", "E-commerce"],
        "value_prop": "Connect with 200+ founders, investors, and mentors in an evening of meaningful conversations"
    },
    "ai_workshop": {
        "name": "AI Implementation for Business Leaders",
        "price": 14999,
        "early_bird": 11999,
        "date": "10-11 July 2024",
        "venue": "Hyderabad HITEC City",
        "capacity": 150,
        "speakers": [
            "AI researchers from IITs",
            "Startup CTOs",
            "Industry practitioners"
        ],
        "topics": [
            "AI strategy for your business",
            "Implementation roadmap",
            "Cost vs ROI analysis",
            "Case studies"
        ],
        "target_industries": ["AI/ML", "Technology", "SaaS", "FinTech"],
        "value_prop": "Learn how to implement AI in your business without wasting crores on failed experiments"
    }
}

print(f"✅ Loaded {len(EVENTS)} events:")
for event_name, event_data in EVENTS.items():
    print(f"   • {event_name.title()}: {event_data['name']} (₹{event_data['price']:,})")

# Create sample campaign
sample_campaign = {
    'name': 'AI & SaaS Growth Summit 2024',
    'event_type': 'conference',
    'status': 'active',
    'budget': 50000,
    'target_audience': {
        'industries': ['SaaS', 'AI/ML', 'Technology'],
        'locations': ['Mumbai', 'Bangalore', 'Delhi NCR'],
        'min_score': 70
    }
}

campaign_id = db.save_campaign(sample_campaign)
print(f"\n✅ Sample campaign created with ID: {campaign_id}")


📅 LOADING EVENTS CATALOG
✅ Loaded 5 events:
   • Conference: AI & SaaS Growth Summit 2024 (₹4,999)
   • Workshop: Growth Hacking Masterclass (₹9,999)
   • Mastermind: CEO Peer Group (12-Month Program) (₹49,999)
   • Networking: Startup Founders Connect (₹999)
   • Ai_Workshop: AI Implementation for Business Leaders (₹14,999)

✅ Sample campaign created with ID: CAMP1772639792


/tmp/ipykernel_363/777518129.py:168: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.cursor.execute("""


In [ ]:
# ==============================================
# CELL 7: EMAIL SENDER
# ==============================================

print("\n" + "="*60)
print("📧 INITIALIZING EMAIL SENDER")
print("="*60)

class EmailSender:
    """Send emails using built-in smtplib"""

    def __init__(self):
        self.smtp_server = "smtp.gmail.com"
        self.port = 587
        self.sender_email = None
        self.password = None
        self.setup_email()

    def setup_email(self):
        """Setup email credentials"""
        print("\n📧 Email Setup Instructions:")
        print("1. Use Gmail App Password (not your regular password)")
        print("2. Get it from: https://myaccount.google.com/apppasswords")
        print("3. Enable 2-Factor Authentication first if not enabled")
        print("\n⚠️  You can skip this now and use MOCK mode for testing")

        choice = input("\nDo you want to configure email now? (y/n): ").lower()

        if choice == 'y':
            self.sender_email = input("Enter your Gmail address: ")
            self.password = getpass.getpass("Enter your Gmail App Password: ")

            if self.sender_email and self.password:
                print("✅ Email configured for:", self.sender_email)
                self.configured = True
            else:
                print("⚠️ Email not configured. Using mock mode.")
                self.configured = False
        else:
            print("📧 Using MOCK mode (no actual emails will be sent)")
            self.configured = False

    def send_email(self, to_email: str, subject: str, body: str) -> bool:
        """Send email using smtplib"""

        if not self.configured:
            print(f"\n📧 [MOCK] Email to: {to_email}")
            print(f"   Subject: {subject}")
            print(f"   Body preview: {body[:150]}...")
            return True

        try:
            # Create message
            msg = MIMEMultipart()
            msg['From'] = self.sender_email
            msg['To'] = to_email
            msg['Subject'] = subject
            msg.attach(MIMEText(body, 'plain'))

            # Send via SMTP
            server = smtplib.SMTP(self.smtp_server, self.port)
            server.starttls()
            server.login(self.sender_email, self.password)
            server.send_message(msg)
            server.quit()

            print(f"✅ Email sent to {to_email}")
            return True

        except Exception as e:
            print(f"❌ Failed to send email: {e}")
            return False

    def send_bulk(self, prospects: List[Dict], subject: str, body_template: str) -> List[Dict]:
        """Send bulk emails"""
        results = []
        for prospect in prospects:
            # Personalize body
            personalized_body = body_template.replace("{name}", prospect.get('name', 'there'))

            success = self.send_email(
                prospect.get('email', ''),
                subject,
                personalized_body
            )

            results.append({
                'prospect': prospect['name'],
                'email': prospect.get('email'),
                'success': success
            })

            # Small delay to avoid rate limits
            time.sleep(1)

        return results

# Initialize email sender
email_sender = EmailSender()


📧 INITIALIZING EMAIL SENDER

📧 Email Setup Instructions:
1. Use Gmail App Password (not your regular password)
2. Get it from: https://myaccount.google.com/apppasswords
3. Enable 2-Factor Authentication first if not enabled

⚠️  You can skip this now and use MOCK mode for testing

Do you want to configure email now? (y/n): y
Enter your Gmail address: sarveshwar.ds.01@gmail.com
Enter your Gmail App Password: ··········
✅ Email configured for: sarveshwar.ds.01@gmail.com


In [ ]:
# ==============================================
# CELL 8: MESSAGE GENERATOR
# ==============================================

print("\n" + "="*60)
print("✍️ INITIALIZING MESSAGE GENERATOR")
print("="*60)

class MessageGenerator:
    """Generate personalized outreach messages"""

    def __init__(self):
        self.templates_used = 0
        print("✅ Message generator ready")

    def generate(self, prospect: Dict, event: Dict) -> str:
        """Generate personalized message"""

        self.templates_used += 1

        # Personalization elements
        templates = [
            f"""Hi {prospect.get('name', 'there')} 👋,

I noticed you're the {prospect.get('title', 'leader')} at {prospect.get('company', 'your company')} and your work in {prospect.get('industry', 'business')} is impressive.

Given your focus on {prospect.get('industry', '')}, I thought you'd be interested in our upcoming {event.get('name')} on {event.get('date')} at {event.get('venue')}.

We have amazing speakers like {event.get('speakers', ['industry leaders'])[0]} and {event.get('speakers', ['experts'])[1] if len(event.get('speakers', [])) > 1 else 'successful founders'} sharing insights on topics relevant to leaders like you.

Would you be open to learning more? I can share the complete agenda and speaker lineup.

Ticket price: ₹{event.get('price', 0):,} (Early bird: ₹{event.get('early_bird', 0):,})

Best regards,
EventFlow AI Team""",

            f"""Hello {prospect.get('name', 'there')},

I've been following {prospect.get('company', 'your company')}'s journey in the {prospect.get('industry', '')} space, and I'm impressed by your growth.

As a {prospect.get('title', 'leader')}, you're probably dealing with challenges around growth and scaling. Our {event.get('name')} on {event.get('date')} brings together leaders who've solved exactly these challenges.

The event features:
• {event.get('speakers', ['Industry leaders'])[0]} - sharing proven frameworks
• {event.get('speakers', ['Experts'])[1] if len(event.get('speakers', [])) > 1 else 'Successful founders'} - insights on scaling
• 50+ industry leaders for networking

Early bird tickets at ₹{event.get('early_bird', 0):,} (Regular: ₹{event.get('price', 0):,})

Worth a 5-minute chat?

Thanks,
EventFlow AI""",

            f"""Hi {prospect.get('name', 'there')},

I saw that {prospect.get('company', 'your company')} is doing amazing work in {prospect.get('industry', '')} and thought you might be interested in our upcoming event.

At {event.get('name')} on {event.get('date')}, we're bringing together {prospect.get('industry', '')} leaders to tackle challenges like:
• Scaling challenges
• Growth strategies
• Fundraising

With speakers like {event.get('speakers', ['industry leaders'])[0]} and {event.get('speakers', ['experts'])[1] if len(event.get('speakers', [])) > 1 else 'successful founders'}, it's going to be incredibly valuable.

Would you like to see if this aligns with your goals?

Ticket info: ₹{event.get('price', 0):,} (Early bird: ₹{event.get('early_bird', 0):,})

Cheers,
EventFlow AI"""
        ]

        import random
        return random.choice(templates)

# Initialize message generator
msg_gen = MessageGenerator()

# Test message generation
test_prospect = INDIAN_PROSPECTS[0]
test_event = EVENTS['conference']
test_message = msg_gen.generate(test_prospect, test_event)

print("\n📝 Sample Message Preview:")
print("-" * 40)
print(test_message[:300] + "...")
print("-" * 40)


✍️ INITIALIZING MESSAGE GENERATOR
✅ Message generator ready

📝 Sample Message Preview:
----------------------------------------
Hi Rahul Sharma,

I saw that TechStart India is doing amazing work in SaaS and thought you might be interested in our upcoming event.

At AI & SaaS Growth Summit 2024 on 15-16 June 2024, we're bringing together SaaS leaders to tackle challenges like:
• Scaling challenges
• Growth strategies
• Fundra...
----------------------------------------


In [ ]:
# ==============================================
# CELL 9: LEAD QUALIFIER (BANT FRAMEWORK)
# ==============================================

print("\n" + "="*60)
print("⭐ INITIALIZING LEAD QUALIFIER")
print("="*60)

class LeadQualifier:
    """Qualify leads using BANT framework"""

    def __init__(self):
        self.qualified_leads = 0
        self.conversions = 0
        print("✅ Lead qualifier ready")

    def qualify(self, prospect: Dict) -> Dict:
        """Qualify a lead using BANT"""

        # Calculate BANT scores
        bant = {
            "Budget": 0,
            "Authority": 0,
            "Need": 0,
            "Timeline": 0
        }

        # Budget score based on company revenue
        revenue_str = prospect.get('revenue', '₹0')
        try:
            revenue_val = float(revenue_str.replace('₹', '').replace('Cr', ''))
            if revenue_val >= 10:
                bant["Budget"] = random.randint(85, 100)
            elif revenue_val >= 5:
                bant["Budget"] = random.randint(70, 90)
            elif revenue_val >= 2:
                bant["Budget"] = random.randint(50, 75)
            else:
                bant["Budget"] = random.randint(30, 55)
        except:
            bant["Budget"] = random.randint(50, 80)

        # Authority based on title
        title = prospect.get('title', '')
        if any(t in title for t in ['CEO', 'Founder', 'CTO', 'Owner']):
            bant["Authority"] = random.randint(90, 100)
        elif any(t in title for t in ['VP', 'Director', 'Head']):
            bant["Authority"] = random.randint(75, 90)
        elif any(t in title for t in ['Manager', 'Lead']):
            bant["Authority"] = random.randint(60, 80)
        else:
            bant["Authority"] = random.randint(40, 60)

        # Need based on engagement score
        engagement = prospect.get('engagement_score', 50)
        bant["Need"] = random.randint(max(50, engagement - 10), min(100, engagement + 10))

        # Timeline based on engagement
        if engagement > 80:
            bant["Timeline"] = random.randint(75, 95)
        elif engagement > 60:
            bant["Timeline"] = random.randint(50, 80)
        else:
            bant["Timeline"] = random.randint(30, 60)

        overall = sum(bant.values()) / 4

        # Determine status
        if overall >= 80:
            status = "🔥 HOT LEAD"
            next_action = "Send payment link & schedule call"
            probability = "85-95%"
        elif overall >= 65:
            status = "⭐ WARM LEAD"
            next_action = "Schedule follow-up in 3 days"
            probability = "50-70%"
        elif overall >= 50:
            status = "🌱 COOL LEAD"
            next_action = "Add to nurture sequence"
            probability = "20-40%"
        else:
            status = "❄️ COLD LEAD"
            next_action = "Newsletter & long-term nurture"
            probability = "<10%"

        return {
            "name": prospect.get('name', ''),
            "title": prospect.get('title', ''),
            "company": prospect.get('company', ''),
            "bant_scores": bant,
            "overall_score": round(overall, 1),
            "status": status,
            "next_action": next_action,
            "probability": probability,
            "engagement_score": prospect.get('engagement_score', 50)
        }

# Initialize qualifier
qualifier = LeadQualifier()

# Test qualification
print("\n📊 Sample Qualification:")
test_result = qualifier.qualify(test_prospect)
print(f"   Prospect: {test_result['name']} - {test_result['company']}")
print(f"   Score: {test_result['overall_score']}%")
print(f"   Status: {test_result['status']}")
print(f"   Next: {test_result['next_action']}")


⭐ INITIALIZING LEAD QUALIFIER
✅ Lead qualifier ready

📊 Sample Qualification:
   Prospect: Rahul Sharma - TechStart India
   Score: 84.2%
   Status: 🔥 HOT LEAD
   Next: Send payment link & schedule call


In [ ]:
# ==============================================
# CELL 10: RUN COMPLETE CAMPAIGN
# ==============================================

print("\n" + "🚀"*60)
print("🚀 RUNNING COMPLETE EVENTFLOW AI CAMPAIGN")
print("🚀"*60)

def run_campaign(event_type="conference", num_prospects=5):
    """Run a complete campaign"""

    print(f"\n📅 Event: {EVENTS[event_type]['name']}")
    print(f"💰 Price: ₹{EVENTS[event_type]['price']:,}")
    print(f"📍 Venue: {EVENTS[event_type]['venue']}")
    print(f"📊 Targeting top {num_prospects} prospects...\n")

    # Get prospects from database
    all_prospects = db.get_all_prospects()
    prospects = all_prospects[:num_prospects]

    results = {
        'event': event_type,
        'prospects_contacted': 0,
        'messages_sent': [],
        'qualifications': [],
        'payments': []
    }

    # Process each prospect
    for i, prospect_tuple in enumerate(prospects, 1):
        # Convert tuple to dict
        prospect = {
            'id': prospect_tuple[0],
            'name': prospect_tuple[1],
            'title': prospect_tuple[2],
            'company': prospect_tuple[3],
            'industry': prospect_tuple[4],
            'location': prospect_tuple[5],
            'email': prospect_tuple[6],
            'engagement_score': prospect_tuple[10]
        }

        print(f"\n{'='*40}")
        print(f"📋 Prospect {i}/{num_prospects}: {prospect['name']}")
        print(f"   {prospect['title']} at {prospect['company']}")
        print(f"   Industry: {prospect['industry']} | Location: {prospect['location']}")

        # Generate message
        message = msg_gen.generate(prospect, EVENTS[event_type])

        # Send email (mock)
        email_sender.send_email(
            prospect.get('email', 'test@example.com'),
            f"Invitation: {EVENTS[event_type]['name']}",
            message
        )

        # Save to database
        msg_id = db.save_message(prospect['id'], f"CAMP_{event_type}", message)
        results['messages_sent'].append({'prospect': prospect['name'], 'msg_id': msg_id})

        # Qualify lead
        qualification = qualifier.qualify(prospect)
        results['qualifications'].append(qualification)

        print(f"   BANT Score: {qualification['overall_score']}% - {qualification['status']}")

        # Create payment link for hot leads
        if qualification['overall_score'] >= 80:
            payment_link = f"https://rzp.io/i/TEST{random.randint(1000,9999)}"
            payment_id = db.save_payment(prospect['id'], f"CAMP_{event_type}", EVENTS[event_type]['price'], payment_link)
            results['payments'].append({'prospect': prospect['name'], 'payment_id': payment_id})
            print(f"   💰 Payment link created")

        results['prospects_contacted'] += 1

    # Summary
    print(f"\n{'='*60}")
    print("📊 CAMPAIGN SUMMARY")
    print(f"{'='*60}")
    print(f"✅ Prospects Contacted: {results['prospects_contacted']}")
    print(f"✅ Messages Sent: {len(results['messages_sent'])}")

    hot_leads = sum(1 for q in results['qualifications'] if q['overall_score'] >= 80)
    warm_leads = sum(1 for q in results['qualifications'] if 65 <= q['overall_score'] < 80)

    print(f"🔥 Hot Leads: {hot_leads}")
    print(f"⭐ Warm Leads: {warm_leads}")
    print(f"💰 Payment Links Created: {len(results['payments'])}")

    # Projected revenue
    projected_revenue = hot_leads * EVENTS[event_type]['price'] * 0.3  # 30% conversion
    print(f"📈 Projected Revenue: ₹{projected_revenue:,.0f}")

    return results

# Run the campaign
print("\n🎯 Choose campaign type:")
print("1. Conference")
print("2. Workshop")
print("3. Mastermind")
print("4. Networking")
print("5. AI Workshop")

choice = input("\nEnter your choice (1-5): ")

event_map = {
    '1': 'conference',
    '2': 'workshop',
    '3': 'mastermind',
    '4': 'networking',
    '5': 'ai_workshop'
}

if choice in event_map:
    results = run_campaign(event_map[choice], num_prospects=5)
else:
    print("❌ Invalid choice. Running default conference campaign.")
    results = run_campaign('conference', num_prospects=5)

print("\n" + "🎉"*60)
print("🎉 CAMPAIGN COMPLETED SUCCESSFULLY!")
print("🎉"*60)

# Show database stats
print("\n📊 FINAL DATABASE STATS:")
stats = db.get_campaign_stats(f"CAMP_{event_map.get(choice, 'conference')}")
print(f"   Total Contacted: {stats.get('total_contacted', 0)}")
print(f"   Total Prospects: {stats.get('total_prospects', 0)}")
print(f"   Total Revenue: ₹{stats.get('total_revenue', 0):,}")


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
🚀 RUNNING COMPLETE EVENTFLOW AI CAMPAIGN
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

🎯 Choose campaign type:
1. Conference
2. Workshop
3. Mastermind
4. Networking
5. AI Workshop

Enter your choice (1-5): 3

📅 Event: CEO Peer Group (12-Month Program)
💰 Price: ₹49,999
📍 Venue: Multiple cities (Mumbai, Bangalore, Delhi)
📊 Targeting top 5 prospects...


📋 Prospect 1/5: Priya Patel
   Founder at AI Solutions
   Industry: AI/ML | Location: Bangalore
❌ Failed to send email: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials a1e0cc1a2514c-94df641e133sm18459242241.5 - gsmtp')
   BANT Score: 74.8% - ⭐ WARM LEAD

📋 Prospect 2/5: Rahul Sharma
   CEO at TechStart India
   Industry: SaaS | Location: Mumbai


/tmp/ipykernel_363/777518129.py:192: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.cursor.execute("""


❌ Failed to send email: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials a1e0cc1a2514c-94df6577a4fsm19553788241.10 - gsmtp')
   BANT Score: 80.2% - 🔥 HOT LEAD
   💰 Payment link created

📋 Prospect 3/5: Rahul Sharma
   CEO at TechStart India
   Industry: SaaS | Location: Mumbai


/tmp/ipykernel_363/777518129.py:206: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.cursor.execute("""


❌ Failed to send email: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials a1e0cc1a2514c-94df6417a98sm19164239241.4 - gsmtp')
   BANT Score: 77.8% - ⭐ WARM LEAD

📋 Prospect 4/5: Kavita Krishnan
   Founder at DataMatic
   Industry: AI/ML | Location: Chennai
❌ Failed to send email: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials 71dfb90a1353d-56ae16a4e31sm4143956e0c.11 - gsmtp')
   BANT Score: 79.8% - ⭐ WARM LEAD

📋 Prospect 5/5: Neha Singh
   Director at InnovateLabs
   Industry: Technology | Location: Pune
❌ Failed to send email: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials ada2fe7eead31-5ffa300c3f8sm3417483137.6 - gsmtp')
   BANT Score: 72.2% - ⭐ WARM LEAD

📊 CAMPAIGN SUMMARY
✅ Prospects Contacted: 5
✅ Messages Sent: 5
🔥 Hot 

In [ ]:
# ==============================================
# CELL 11: ANALYTICS DASHBOARD
# ==============================================

print("\n" + "="*60)
print("📊 ANALYTICS DASHBOARD")
print("="*60)

class AnalyticsDashboard:
    """Simple analytics dashboard"""

    def __init__(self, db):
        self.db = db

    def show_summary(self):
        """Show summary statistics"""

        all_prospects = self.db.get_all_prospects()

        print(f"\n📈 OVERALL METRICS:")
        print(f"   Total Prospects: {len(all_prospects)}")

        # Industry breakdown
        industries = {}
        locations = {}
        for p in all_prospects:
            ind = p[4]  # industry
            loc = p[5]  # location
            industries[ind] = industries.get(ind, 0) + 1
            locations[loc] = locations.get(loc, 0) + 1

        print(f"\n🏢 INDUSTRY BREAKDOWN:")
        for ind, count in sorted(industries.items(), key=lambda x: x[1], reverse=True):
            print(f"   • {ind}: {count} prospects")

        print(f"\n📍 LOCATION BREAKDOWN:")
        for loc, count in sorted(locations.items(), key=lambda x: x[1], reverse=True):
            print(f"   • {loc}: {count} prospects")

        # Campaign stats
        print(f"\n📊 CAMPAIGN PERFORMANCE:")
        for event_type in EVENTS.keys():
            stats = self.db.get_campaign_stats(f"CAMP_{event_type}")
            if stats.get('total_contacted', 0) > 0:
                print(f"\n   {event_type.title()}:")
                print(f"      Contacted: {stats['total_contacted']}")
                print(f"      Revenue: ₹{stats['total_revenue']:,.0f}")

# Show analytics
analytics = AnalyticsDashboard(db)
analytics.show_summary()

print("\n✅ Analytics ready!")


📊 ANALYTICS DASHBOARD

📈 OVERALL METRICS:
   Total Prospects: 11

🏢 INDUSTRY BREAKDOWN:
   • AI/ML: 2 prospects
   • SaaS: 2 prospects
   • Technology: 2 prospects
   • Enterprise SaaS: 1 prospects
   • FinTech: 1 prospects
   • Cloud Computing: 1 prospects
   • HR Tech: 1 prospects
   • Marketing: 1 prospects

📍 LOCATION BREAKDOWN:
   • Mumbai: 3 prospects
   • Bangalore: 2 prospects
   • Chennai: 2 prospects
   • Pune: 1 prospects
   • Gurgaon: 1 prospects
   • Delhi NCR: 1 prospects
   • Hyderabad: 1 prospects

📊 CAMPAIGN PERFORMANCE:

✅ Analytics ready!


In [ ]:
# ==============================================
# CELL 12: CLEAN UP
# ==============================================

print("\n" + "="*60)
print("🧹 CLEANING UP")
print("="*60)

# Close database connection
db.close()

print("\n✅ All done! EventFlow AI system ready.")
print("\n📝 Next steps:")
print("1. Add your OpenAI API key for AI-generated messages")
print("2. Configure Gmail for real email sending")
print("3. Set up Razorpay for actual payments")
print("4. Deploy on cloud for 24/7 operation")


🧹 CLEANING UP
✅ Database connection closed

✅ All done! EventFlow AI system ready.

📝 Next steps:
1. Add your OpenAI API key for AI-generated messages
2. Configure Gmail for real email sending
3. Set up Razorpay for actual payments
4. Deploy on cloud for 24/7 operation


In [ ]:
# ==============================================
# CELL 13: DAILY AUTO-CAMPAIGN SCHEDULER
# ==============================================

!pip install schedule

print("\n" + "="*60)
print("⏰ DAILY AUTO-CAMPAIGN SCHEDULER")
print("="*60)

import schedule
import time
import threading
from datetime import datetime

class CampaignScheduler:
    """Automatically run campaigns on schedule"""

    def __init__(self, db, msg_gen, email_sender, qualifier):
        self.db = db
        self.msg_gen = msg_gen
        self.email_sender = email_sender
        self.qualifier = qualifier
        self.running = False
        self.scheduled_jobs = []
        print("✅ Scheduler initialized")

    def morning_campaign(self):
        """Run morning campaign at 9 AM"""
        print(f"\n{'='*60}")
        print(f"🌅 MORNING CAMPAIGN - {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f"{'='*60}")

        # Get new prospects (engagement score > 70)
        all_prospects = self.db.get_all_prospects()
        new_prospects = [p for p in all_prospects if p[10] > 70][:5]  # Top 5 hot prospects

        if new_prospects:
            self.run_scheduled_campaign(new_prospects, "conference", "Morning Outreach")
        else:
            print("📭 No new hot prospects found")

    def afternoon_followup(self):
        """Send follow-ups at 2 PM"""
        print(f"\n{'='*60}")
        print(f"☀️ AFTERNOON FOLLOW-UPS - {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f"{'='*60}")

        # Get prospects who haven't replied
        # This is simplified - in real system, track replies in database
        all_prospects = self.db.get_all_prospects()
        followup_prospects = all_prospects[:3]  # Top 3 for demo

        for p in followup_prospects:
            prospect = {
                'name': p[1], 'email': p[6], 'company': p[3],
                'title': p[2], 'industry': p[4]
            }

            followup_msg = f"""
Hi {prospect['name']} 👋,

Just checking if you saw my previous email about the upcoming event.

We still have early bird tickets available at ₹{EVENTS['conference']['early_bird']:,} and I'd love to have you join us.

Would you like me to send the full agenda?

Best regards,
EventFlow AI
"""
            self.email_sender.send_email(
                prospect['email'],
                "Following up: AI & SaaS Summit",
                followup_msg
            )
            print(f"✅ Follow-up sent to {prospect['name']}")

    def evening_report(self):
        """Send daily report at 6 PM"""
        print(f"\n{'='*60}")
        print(f"🌙 EVENING REPORT - {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f"{'='*60}")

        # Generate report
        all_prospects = self.db.get_all_prospects()

        report = f"""
📊 DAILY CAMPAIGN REPORT
{datetime.now().strftime('%Y-%m-%d')}

✅ Total Prospects: {len(all_prospects)}
🔥 Hot Leads: {sum(1 for p in all_prospects if p[10] > 80)}
⭐ Warm Leads: {sum(1 for p in all_prospects if 60 < p[10] <= 80)}
🌱 Cold Leads: {sum(1 for p in all_prospects if p[10] <= 60)}

📈 Today's Activity:
• Emails Sent: {len(all_prospects)}
• Follow-ups: 3
• Responses: Pending

🎯 Tomorrow's Targets:
• Send 10 new emails
• Follow up with 5 hot leads
• Qualify new prospects
"""
        print(report)

        # Save report to file
        with open(f"report_{datetime.now().strftime('%Y%m%d')}.txt", 'w') as f:
            f.write(report)
        print("✅ Report saved")

    def run_scheduled_campaign(self, prospects, event_type="conference", campaign_name="Scheduled"):
        """Run campaign for scheduled prospects"""

        print(f"\n📅 Running {campaign_name}...")

        for prospect_tuple in prospects:
            prospect = {
                'id': prospect_tuple[0],
                'name': prospect_tuple[1],
                'title': prospect_tuple[2],
                'company': prospect_tuple[3],
                'industry': prospect_tuple[4],
                'location': prospect_tuple[5],
                'email': prospect_tuple[6],
                'engagement_score': prospect_tuple[10]
            }

            # Generate and send message
            message = self.msg_gen.generate(prospect, EVENTS[event_type])
            self.email_sender.send_email(
                prospect['email'],
                f"Invitation: {EVENTS[event_type]['name']}",
                message
            )

            # Save to database
            self.db.save_message(prospect['id'], f"AUTO_{event_type}", message)

            # Qualify
            qualification = self.qualifier.qualify(prospect)
            print(f"   • {prospect['name']}: {qualification['status']}")

    def start_scheduler(self):
        """Start the scheduler in background thread"""

        # Schedule jobs
        schedule.every().day.at("09:00").do(self.morning_campaign)
        schedule.every().day.at("14:00").do(self.afternoon_followup)
        schedule.every().day.at("18:00").do(self.evening_report)

        # Also run every hour for demo (in Colab)
        schedule.every(1).minutes.do(self.morning_campaign)  # For testing only

        self.running = True
        print("\n⏰ Scheduler started! Running every minute for demo...")
        print("(In production, it will run at 9 AM, 2 PM, 6 PM)")

        def run_scheduler():
            while self.running:
                schedule.run_pending()
                time.sleep(1)

        # Start in background thread
        thread = threading.Thread(target=run_scheduler, daemon=True)
        thread.start()
        return thread

    def stop_scheduler(self):
        """Stop the scheduler"""
        self.running = False
        print("⏹️ Scheduler stopped")

# Initialize scheduler
scheduler = CampaignScheduler(db, msg_gen, email_sender, qualifier)

# Start scheduler (for testing - runs every minute)
scheduler_thread = scheduler.start_scheduler()

print("\n✅ Scheduler running in background!")
print("Watch for automated campaigns every minute...")


⏰ DAILY AUTO-CAMPAIGN SCHEDULER
✅ Scheduler initialized

⏰ Scheduler started! Running every minute for demo...
(In production, it will run at 9 AM, 2 PM, 6 PM)

✅ Scheduler running in background!
Watch for automated campaigns every minute...


In [ ]:
# ==============================================
# CELL 14: WHATSAPP INTEGRATION
# ==============================================

print("\n" + "="*60)
print("📱 WHATSAPP BUSINESS INTEGRATION")
print("="*60)

class WhatsAppSender:
    """Send WhatsApp messages (mock for now)"""

    def __init__(self):
        # In production, use WhatsApp Business API
        self.account_sid = None
        self.auth_token = None
        self.from_number = None
        self.setup_whatsapp()

    def setup_whatsapp(self):
        """Setup WhatsApp credentials"""
        print("\n📱 WhatsApp Setup (Optional):")
        print("Use Twilio or WhatsApp Business API for real integration")
        print("For now, using mock mode...")

        choice = input("\nConfigure WhatsApp now? (y/n): ").lower()

        if choice == 'y':
            self.account_sid = input("Enter Twilio Account SID: ")
            self.auth_token = getpass.getpass("Enter Auth Token: ")
            self.from_number = input("Enter WhatsApp number (with country code): ")
            print("✅ WhatsApp configured!")
        else:
            print("📱 Using mock WhatsApp mode")

    def send_message(self, to_number: str, message: str) -> bool:
        """Send WhatsApp message"""

        if not self.account_sid:
            print(f"\n📱 [MOCK] WhatsApp to {to_number}")
            print(f"   Message: {message[:100]}...")
            return True

        try:
            # In production, use Twilio or WhatsApp Business API
            # from twilio.rest import Client
            # client = Client(self.account_sid, self.auth_token)
            # client.messages.create(
            #     body=message,
            #     from_=f'whatsapp:{self.from_number}',
            #     to=f'whatsapp:{to_number}'
            # )
            print(f"✅ WhatsApp sent to {to_number}")
            return True
        except Exception as e:
            print(f"❌ Failed: {e}")
            return False

    def broadcast(self, prospects: List[Dict], message_template: str):
        """Send bulk WhatsApp messages"""
        results = []
        for p in prospects:
            if p.get('phone'):
                personalized = message_template.replace("{name}", p['name'])
                success = self.send_message(p['phone'], personalized)
                results.append({'name': p['name'], 'success': success})
                time.sleep(2)  # Avoid rate limits
        return results

# Initialize WhatsApp
whatsapp = WhatsAppSender()

# Test WhatsApp
test_prospect = INDIAN_PROSPECTS[0]
test_msg = f"Hi {test_prospect['name']}! Check out our AI & SaaS Summit!"
whatsapp.send_message(test_prospect['phone'], test_msg)


📱 WHATSAPP BUSINESS INTEGRATION

📱 WhatsApp Setup (Optional):
Use Twilio or WhatsApp Business API for real integration
For now, using mock mode...

Configure WhatsApp now? (y/n): y
Enter Twilio Account SID: 1232
Enter Auth Token: ··········


Exception in thread Thread-3 (run_scheduler):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_363/1467548377.py", line 163, in run_scheduler
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 854, in run_pending
    default_scheduler.run_pending()
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 101, in run_pending
    self._run_job(job)
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 173, in _run_job
    ret = job.run()
          ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 691, in run
    ret = self.job_func()
          ^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_363/1467548377.py", line 35, in morning_campaign
  File "/tmp/ipykernel_363/777518129.py", line 261, in ge


🌅 MORNING CAMPAIGN - 2026-03-04 16:07
Enter WhatsApp number (with country code): +917439071619
✅ WhatsApp configured!
✅ WhatsApp sent to +91 98765 43210


True

In [ ]:
# ==============================================
# CELL 15: SMS MARKETING
# ==============================================

print("\n" + "="*60)
print("📲 SMS MARKETING INTEGRATION")
print("="*60)

class SMSSender:
    """Send SMS messages"""

    def __init__(self):
        self.provider = "MSG91"  # or Twilio
        self.api_key = None
        self.sender_id = "EVENTFLOW"
        self.setup_sms()

    def setup_sms(self):
        """Setup SMS credentials"""
        print("\n📲 SMS Setup (Optional):")
        print("Use MSG91 or Twilio for real SMS")

        choice = input("\nConfigure SMS now? (y/n): ").lower()

        if choice == 'y':
            self.api_key = getpass.getpass("Enter API Key: ")
            print("✅ SMS configured!")
        else:
            print("📲 Using mock SMS mode")

    def send_sms(self, to_number: str, message: str) -> bool:
        """Send SMS message"""

        if not self.api_key:
            print(f"\n📲 [MOCK] SMS to {to_number}")
            print(f"   Message: {message[:100]}...")
            return True

        try:
            # In production, use MSG91 or Twilio API
            # import requests
            # url = f"https://api.msg91.com/api/sendhttp.php"
            # params = {
            #     'authkey': self.api_key,
            #     'mobiles': to_number,
            #     'message': message,
            #     'sender': self.sender_id,
            #     'route': 4
            # }
            # response = requests.get(url, params=params)
            print(f"✅ SMS sent to {to_number}")
            return True
        except Exception as e:
            print(f"❌ Failed: {e}")
            return False

    def send_reminder(self, prospect, event, days_before=1):
        """Send event reminder SMS"""
        reminder = f"""
Reminder: {event['name']} is tomorrow!
Date: {event['date']}
Venue: {event['venue']}
See you there! - EventFlow AI
"""
        return self.send_sms(prospect['phone'], reminder)

# Initialize SMS
sms = SMSSender()

# Test SMS
test_prospect = INDIAN_PROSPECTS[1]
sms.send_reminder(test_prospect, EVENTS['conference'])


📲 SMS MARKETING INTEGRATION

📲 SMS Setup (Optional):
Use MSG91 or Twilio for real SMS

Configure SMS now? (y/n): y
Enter API Key: ··········
✅ SMS configured!
✅ SMS sent to +91 98765 43211


True

In [ ]:
# ==============================================
# CELL 16: LEAD SCORING WITH MACHINE LEARNING
# ==============================================

!pip install scikit-learn

print("\n" + "="*60)
print("🤖 ML LEAD SCORING MODEL")
print("="*60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

class MLLeadScorer:
    """Machine Learning model for lead scoring"""

    def __init__(self):
        self.model = None
        self.label_encoders = {}
        self.feature_columns = ['title_score', 'industry_score', 'location_score',
                                'company_size_score', 'engagement_score']
        print("✅ ML Scorer initialized")

    def prepare_training_data(self, prospects_data):
        """Prepare data for training"""

        df = pd.DataFrame(prospects_data)

        # Create features
        df['title_score'] = df['title'].apply(self.encode_title)
        df['industry_score'] = df['industry'].apply(self.encode_industry)
        df['location_score'] = df['location'].apply(self.encode_location)
        df['company_size_score'] = df['company_size'].apply(self.encode_company_size)

        # Target variable (1 = converted, 0 = not converted)
        # For demo, create synthetic target
        df['converted'] = np.random.choice([0, 1], size=len(df), p=[0.7, 0.3])

        return df

    def encode_title(self, title):
        """Encode title to score"""
        title_scores = {
            'CEO': 100, 'Founder': 95, 'CTO': 90, 'Director': 85,
            'VP': 85, 'Head': 80, 'Manager': 70, 'Other': 50
        }
        for key in title_scores:
            if key in str(title):
                return title_scores[key]
        return 50

    def encode_industry(self, industry):
        """Encode industry to score"""
        industry_scores = {
            'SaaS': 90, 'AI/ML': 95, 'FinTech': 85, 'Technology': 80,
            'Marketing': 70, 'HR Tech': 65, 'Other': 50
        }
        return industry_scores.get(industry, 50)

    def encode_location(self, location):
        """Encode location to score"""
        location_scores = {
            'Mumbai': 90, 'Bangalore': 95, 'Delhi NCR': 85,
            'Pune': 80, 'Hyderabad': 80, 'Chennai': 75
        }
        return location_scores.get(location, 70)

    def encode_company_size(self, size):
        """Encode company size to score"""
        try:
            if '200-500' in size: return 95
            elif '100-250' in size: return 90
            elif '50-200' in size: return 85
            elif '50-150' in size: return 80
            elif '30-100' in size: return 75
            elif '10-50' in size: return 70
            else: return 60
        except:
            return 70

    def train(self, prospects_data):
        """Train the ML model"""

        print("\n🤖 Training ML model...")

        # Prepare data
        df = self.prepare_training_data(prospects_data)

        # Features and target
        X = df[self.feature_columns]
        y = df['converted']

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Train model
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.model.fit(X_train, y_train)

        # Evaluate
        train_score = self.model.score(X_train, y_train)
        test_score = self.model.score(X_test, y_test)

        print(f"✅ Model trained!")
        print(f"   Training accuracy: {train_score:.2%}")
        print(f"   Test accuracy: {test_score:.2%}")

        # Feature importance
        importance = dict(zip(self.feature_columns, self.model.feature_importances_))
        print("\n📊 Feature Importance:")
        for feat, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True):
            print(f"   • {feat}: {imp:.2%}")

        return self.model

    def predict_score(self, prospect):
        """Predict lead score for a single prospect"""

        if not self.model:
            print("⚠️ Model not trained yet")
            return prospect.get('engagement_score', 50)

        # Prepare features
        features = pd.DataFrame([{
            'title_score': self.encode_title(prospect.get('title', '')),
            'industry_score': self.encode_industry(prospect.get('industry', '')),
            'location_score': self.encode_location(prospect.get('location', '')),
            'company_size_score': self.encode_company_size(prospect.get('company_size', '')),
            'engagement_score': prospect.get('engagement_score', 50)
        }])

        # Predict
        prob = self.model.predict_proba(features)[0][1]  # Probability of conversion
        return round(prob * 100, 1)

    def batch_predict(self, prospects):
        """Predict scores for multiple prospects"""
        results = []
        for p in prospects:
            score = self.predict_score(p)
            results.append({
                'name': p.get('name'),
                'company': p.get('company'),
                'ml_score': score,
                'original_score': p.get('engagement_score', 50)
            })
        return sorted(results, key=lambda x: x['ml_score'], reverse=True)

# Initialize ML scorer
ml_scorer = MLLeadScorer()

# Prepare training data from our prospects
prospects_list = []
for p_tuple in db.get_all_prospects():
    prospects_list.append({
        'name': p_tuple[1],
        'title': p_tuple[2],
        'company': p_tuple[3],
        'industry': p_tuple[4],
        'location': p_tuple[5],
        'company_size': p_tuple[7] if len(p_tuple) > 7 else '50-200',
        'engagement_score': p_tuple[10] if len(p_tuple) > 10 else 50
    })

# Train model
ml_scorer.train(prospects_list)

# Test predictions
print("\n📊 ML Score Predictions:")
test_prospects = prospects_list[:3]
for p in test_prospects:
    ml_score = ml_scorer.predict_score(p)
    print(f"   • {p['name']}: {ml_score}% (Original: {p['engagement_score']}%)")


🤖 ML LEAD SCORING MODEL
✅ ML Scorer initialized


ProgrammingError: Cannot operate on a closed database.

In [ ]:
# ==============================================
# CELL 12.5: REOPEN DATABASE (FIX)
# ==============================================

print("\n" + "="*60)
print("🔄 REOPENING DATABASE CONNECTION")
print("="*60)

# Create new database connection
db = Database("eventflow.db")
print("✅ Database reopened successfully!")

# Verify data is still there
all_prospects = db.get_all_prospects()
print(f"📊 Found {len(all_prospects)} prospects in database")

if all_prospects:
    print("\n📋 Sample prospect:")
    print(f"   Name: {all_prospects[0][1]}")
    print(f"   Title: {all_prospects[0][2]}")
    print(f"   Company: {all_prospects[0][3]}")


🔄 REOPENING DATABASE CONNECTION
✅ Connected to eventflow.db
✅ All tables created/verified
✅ Database reopened successfully!
📊 Found 11 prospects in database

📋 Sample prospect:
   Name: Priya Patel
   Title: Founder
   Company: AI Solutions


In [ ]:
# ==============================================
# CELL 20: RUN ALL NEXT-LEVEL FEATURES (FIXED)
# ==============================================

print("\n" + "🚀"*60)
print("🚀 ACTIVATING ALL NEXT-LEVEL FEATURES")
print("🚀"*60)

# Ensure database is open
try:
    db.get_all_prospects()
    print("✅ Database is open and ready")
except:
    print("🔄 Reopening database...")
    db = Database("eventflow.db")

# 1. Daily Auto-Campaign Scheduler
print("\n⏰ Starting Scheduler...")
scheduler = CampaignScheduler(db, msg_gen, email_sender, qualifier)
scheduler_thread = scheduler.start_scheduler()

# 2. WhatsApp Integration
print("\n📱 Initializing WhatsApp...")
whatsapp = WhatsAppSender()

# 3. SMS Marketing
print("\n📲 Initializing SMS...")
sms = SMSSender()

# 4. ML Lead Scoring
print("\n🤖 Training ML Model...")
ml_scorer = MLLeadScorer()

# Prepare training data
prospects_list = []
for p_tuple in db.get_all_prospects():
    prospects_list.append({
        'name': p_tuple[1],
        'title': p_tuple[2],
        'company': p_tuple[3],
        'industry': p_tuple[4],
        'location': p_tuple[5],
        'company_size': p_tuple[7] if len(p_tuple) > 7 else '50-200',
        'engagement_score': p_tuple[10] if len(p_tuple) > 10 else 50
    })

ml_scorer.train(prospects_list)

# Test predictions
print("\n📊 ML Score Predictions:")
for p in prospects_list[:3]:
    ml_score = ml_scorer.predict_score(p)
    print(f"   • {p['name']}: ML:{ml_score}% vs Original:{p['engagement_score']}%")

# 5. CRM Integration
print("\n🔄 Initializing CRM...")
crm = CRMIntegration()

# 6. Reporting
print("\n📊 Generating Reports...")
reporter = ReportGenerator(db)
excel_file = reporter.export_to_excel()
reporter.create_charts()

print("\n" + "✅"*60)
print("✅ ALL NEXT-LEVEL FEATURES ACTIVATED!")
print("✅"*60 + "\n")

print("📝 Features Running:")
print("   • ⏰ Auto-scheduler (every minute for testing)")
print("   • 📱 WhatsApp integration (mock)")
print("   • 📲 SMS marketing (mock)")
print("   • 🤖 ML lead scoring (trained)")
print("   • 🔄 CRM sync (mock)")
print("   • 📊 Reports & charts (generated)")

Exception in thread Thread-4 (run_scheduler):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_363/1467548377.py", line 163, in run_scheduler



🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
🚀 ACTIVATING ALL NEXT-LEVEL FEATURES
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
✅ Database is open and ready

⏰ Starting Scheduler...
✅ Scheduler initialized

⏰ Scheduler started! Running every minute for demo...
(In production, it will run at 9 AM, 2 PM, 6 PM)

🌅 MORNING CAMPAIGN - 2026-03-04 16:10

📱 Initializing WhatsApp...

📱 WhatsApp Setup (Optional):
Use Twilio or WhatsApp Business API for real integration
For now, using mock mode...


  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 854, in run_pending
    default_scheduler.run_pending()
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 101, in run_pending
    self._run_job(job)
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 173, in _run_job
    ret = job.run()
          ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/schedule/__init__.py", line 691, in run
    ret = self.job_func()
          ^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_363/1467548377.py", line 35, in morning_campaign
  File "/tmp/ipykernel_363/777518129.py", line 261, in get_all_prospects
sqlite3.ProgrammingError: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 133135087747072 and this is thread id 133134010721856.


In [ ]:
# ==============================================
# CELL 12.5: REOPEN DATABASE (FIX)
# ==============================================

print("\n" + "="*60)
print("🔄 REOPENING DATABASE CONNECTION")
print("="*60)

# Create new database connection
db = Database("eventflow.db")
print("✅ Database reopened successfully!")

# Verify data is still there
all_prospects = db.get_all_prospects()
print(f"📊 Found {len(all_prospects)} prospects in database")

if all_prospects:
    print("\n📋 Sample prospect:")
    print(f"   Name: {all_prospects[0][1]}")
    print(f"   Title: {all_prospects[0][2]}")
    print(f"   Company: {all_prospects[0][3]}")